# Lab 03 — Word2Vec and word embeddings

**Course:** Fundamentals of NLP
**Author:** Aymen Ben Brik

## Objectives

1. Train a Word2Vec model on a real corpus and inspect the resulting vectors.
2. Explore semantic similarity, nearest neighbours, and word analogies.
3. Visualize embeddings in 2D using PCA.
4. Contrast skip-gram and CBOW.
5. Identify limitations: out-of-vocabulary words, polysemy, biases.

## Prerequisites

```bash
pip install gensim nltk scikit-learn matplotlib
```

## 1. Setup and corpus

In [ ]:
import nltk
for pkg in ('brown', 'punkt', 'punkt_tab'):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
from nltk.corpus import brown
from gensim.models import Word2Vec
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Brown corpus: ~1M tokens of American English from the 1960s
sentences = [[w.lower() for w in s] for s in brown.sents()]
print(f'Number of sentences: {len(sentences)}')
print(f'First sentence: {sentences[0][:15]} ...')

## 2. Train Word2Vec (skip-gram)

In [ ]:
model_sg = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,            # skip-gram
    negative=5,
    epochs=5,
    workers=4,
    seed=42,
)
print(f'Vocabulary size: {len(model_sg.wv)}')
print(f'Vector dimension: {model_sg.wv.vector_size}')

## 3. Word similarity

In [ ]:
words = ['king', 'queen', 'computer', 'paris', 'love', 'war']
for w in words:
    if w in model_sg.wv:
        sims = model_sg.wv.most_similar(w, topn=5)
        print(f'\nMost similar to "{w}":')
        for sw, s in sims:
            print(f'  {sw:<15} {s:.3f}')
    else:
        print(f'{w} not in vocabulary')

## 4. Word analogies

In [ ]:
def analogy(model, a, b, c):
    """a is to b as c is to ?"""
    try:
        return model.wv.most_similar(positive=[b, c], negative=[a], topn=3)
    except KeyError as e:
        return f'Missing word: {e}'

tests = [
    ('man', 'king', 'woman'),    # king -> queen?
    ('paris', 'france', 'london'),  # london -> england?
    ('walk', 'walking', 'run'),  # run -> running?
]
for a, b, c in tests:
    print(f'{a}:{b} :: {c}:?')
    print(f'  {analogy(model_sg, a, b, c)}')

**Note.** A small corpus (1M tokens) yields imperfect analogies. Production-quality vectors are trained on hundreds of billions of tokens.

## 5. Visualize embeddings (PCA)

In [ ]:
selected = ['king', 'queen', 'man', 'woman', 'paris', 'london', 'tokyo', 'france',
            'england', 'japan', 'apple', 'orange', 'banana', 'computer', 'car', 'truck']
selected = [w for w in selected if w in model_sg.wv]
vectors = np.array([model_sg.wv[w] for w in selected])
vectors_2d = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(8, 6))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1])
for w, (x, y) in zip(selected, vectors_2d):
    plt.annotate(w, (x, y), fontsize=10)
plt.title('Word embeddings projected to 2D (PCA)')
plt.xlabel('PC 1'); plt.ylabel('PC 2')
plt.grid(alpha=0.3); plt.tight_layout()
plt.show()

**Observation.** Geographic words tend to cluster together; royal terms cluster; food terms form their own group. The 100-dim space encodes coarse semantic structure even from a small training corpus.

## 6. Skip-gram vs CBOW

In [ ]:
model_cbow = Word2Vec(
    sentences=sentences,
    vector_size=100, window=5, min_count=5,
    sg=0,            # CBOW
    negative=5, epochs=5, workers=4, seed=42,
)

for w in ['king', 'computer', 'love']:
    print(f'\n--- {w} ---')
    if w in model_sg.wv:
        print('Skip-gram:', [s[0] for s in model_sg.wv.most_similar(w, topn=5)])
    if w in model_cbow.wv:
        print('CBOW    :', [s[0] for s in model_cbow.wv.most_similar(w, topn=5)])

**Discussion.** Skip-gram and CBOW often produce qualitatively different neighbours. Skip-gram tends to be better on rare words; CBOW is faster to train and often better on frequent words.

## 7. Limitations: OOV and polysemy

In [ ]:
# OOV: a word that does not appear at least 5 times in the corpus
for w in ['javascript', 'tunisia', 'transformer']:
    print(f'  {w!r} in vocabulary: {w in model_sg.wv}')

# Polysemy: "bank" has only one vector
if 'bank' in model_sg.wv:
    print("\nMost similar to 'bank' (mixes financial AND river senses):")
    for w, s in model_sg.wv.most_similar('bank', topn=10):
        print(f'  {w:<15} {s:.3f}')

## Exercises

**Exercise 1.** Train Word2Vec with `vector_size=300` instead of 100. Do nearest neighbours improve?

**Exercise 2.** Create three analogy tests of your own (geographic, syntactic, social) and report the results. Discuss which fail and why.

**Exercise 3.** Replace Word2Vec by FastText (`from gensim.models import FastText`). Show that FastText handles OOV words by computing a vector for an unseen word.

**Exercise 4.** Compute the cosine similarity between `'man'` and `'doctor'`, and between `'woman'` and `'doctor'`. Repeat with `'nurse'`. Comment on potential gender bias.

**Exercise 5 (synthesis).** Train Word2Vec on a domain-specific corpus of your choice (e.g., a Wikipedia dump in French, or the NLTK Gutenberg corpus). Compare nearest neighbours of a polysemous word (e.g., *bank*, *bat*, *crane*) with those obtained from the Brown corpus. What does the comparison tell you about the data-dependence of embeddings?